# 3 · Results and cost

Every number of the LoRA experiments, read from the files the training runs wrote: `weights/*.json` and
`weights/bench.json` here, the linear-probe, VPT-deep and full fine-tuning runs from `../vpt/weights`.

In [ ]:
import sys, json, time, math
from pathlib import Path
ROOT = Path.cwd().parent            # demo/lora
VPT = ROOT.parent / "vpt"           # data, linear-probe and full fine-tuning runs
sys.path.insert(0, str(ROOT))
import numpy as np, torch
print("torch", torch.__version__, "| CUDA:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none")

In [ ]:
TASKS = {"dermamnist": "Skin lesion", "bloodmnist": "Blood cell", "pneumoniamnist": "Chest X-ray"}
METHODS = {"linear": (VPT / "weights", "linear"), "vpt": (VPT / "weights", "vpt-deep-p50"),
           "lora": (ROOT / "weights", "lora-r8"), "full": (VPT / "weights", "full")}
R = {(t, m, r): json.loads((d / f"{t}_{tag}_{r}.json").read_text()) for t in TASKS for m, (d, tag) in METHODS.items() for r in ("1k", "full")}
print(f"{'task':12s}{'method':8s}{'images':>8s}{'acc':>8s}{'bal acc':>9s}{'AUC':>8s}{'stored':>11s}")
for t, name in TASKS.items():
    for r in ("1k", "full"):
        for m, (d, tag) in METHODS.items():
            e = R[(t, m, r)]; size = (d / f"{t}_{tag}_{r}.safetensors").stat().st_size
            print(f"{name:12s}{m:8s}{e['train_images']:>8,}{e['test']['acc']:>8.1f}{e['test']['bal_acc']:>9.1f}{e['test']['auc'] / 100:>8.3f}{size / 1e6:>9.2f} MB")
        print()

## Balanced accuracy, two training-set sizes

One group of bars per task and training-set size. LoRA in red, full fine-tuning dark, the two other methods grey.

In [ ]:
import matplotlib.pyplot as plt
colors = {"linear": "#C9CCD0", "vpt": "#9EA3A8", "lora": "#BF3F2A", "full": "#1C1F23"}
fig, axes = plt.subplots(1, 3, figsize=(15, 4.2))
for ax, (t, name) in zip(axes, TASKS.items()):
    for g, r in enumerate(("1k", "full")):
        for j, m in enumerate(METHODS):
            v = R[(t, m, r)]["test"]["bal_acc"]
            ax.bar(g * 5 + j, v, color=colors[m], label=m if g == 0 else None)
            ax.text(g * 5 + j, v + 0.5, f"{v:.1f}", ha="center", fontsize=8)
    ax.set_xticks([1.5, 6.5], ["1,000 images", f"{R[(t, 'lora', 'full')]['train_images']:,} images"])
    ax.set_ylim(0, 105); ax.set_title(name); ax.set_ylabel("test balanced accuracy %")
axes[0].legend(ncol=4, fontsize=8); plt.tight_layout(); plt.show()
for t, name in TASKS.items():
    for r in ("1k", "full"):
        a = {m: R[(t, m, r)]["test"]["bal_acc"] for m in METHODS}
        print(f"{name:12s} {r:4s}  LoRA - VPT {a['lora'] - a['vpt']:+5.1f}   LoRA - linear {a['lora'] - a['linear']:+5.1f}   LoRA - full {a['lora'] - a['full']:+5.1f}")

## What each method costs

`bench.py` measures training memory and time at batch 32 (bf16) and inference on 2 CPU threads, with the same
protocol as the VPT talk's `bench.py`, all four methods in one session. LoRA is timed at inference twice: with the
bypass, and merged.

In [ ]:
b = json.loads((ROOT / "weights" / "bench.json").read_text())
print(f"GPU: {b['gpu']} | batch {b['train_batch']} | CPU threads {b['cpu_threads']}\n")
print(f"{'method':14s}{'train GB':>10s}{'train ms/img':>14s}{'CPU ms/img':>12s}")
for m in ("linear", "vpt", "lora", "full"):
    e = b[m]
    print(f"{m:14s}{e['train_peak_gb']:>10.2f}{e['train_ms_per_img']:>14.2f}{e['cpu_ms_per_img']:>12d}")
print(f"{'lora, merged':14s}{'':>10s}{'':>14s}{b['lora_merged']['cpu_ms_per_img']:>12d}")

## Re-scoring a saved model, with the bypass and merged

Loads the DermaMNIST LoRA file (full split) onto the base model and scores the whole test split, then merges B A into
the weights and scores again. `evaluate.py` does this for all six LoRA models.

In [ ]:
from torch.utils.data import DataLoader
from torchvision import transforms
from safetensors.torch import load_file
from train import build, evaluate
sets, classes = build("dermamnist", transforms.Normalize([0.5] * 3, [0.5] * 3))
dl = DataLoader(sets["test"], batch_size=128, num_workers=0)
from lora_model import load_task
model = load_task(load_file(ROOT / "weights" / "dermamnist_lora-r8_full.safetensors"), len(classes)).cuda()
with torch.no_grad():
    got = evaluate(model, dl, "cuda", len(classes))
    merged = evaluate(model.merge(), dl, "cuda", len(classes))
print("stored    ", R[("dermamnist", "lora", "full")]["test"])
print("re-scored ", got)
print("merged    ", merged)